# Turkish Morph Retrieval — 5-family temiz pilot

Bu notebook `test/data/pilot_verisi_5.json` içindeki **5 query × 11 aday** için hızlı model
karşılaştırması yapar. Kontrollü havuzda `Recall@1/3`, ortak 55-belge full-corpus
retrieval'da `Recall@1/3/10/50` raporlanır.

> Bu veri v3.9 üretim ve iki-judge hattının temiz smoke testidir; küçük örneklem nedeniyle paper sonucu değildir.

In [ ]:
%pip -q install -U "sentence-transformers>=3.0,<6" "transformers>=4.48,<5" pandas matplotlib seaborn

In [ ]:
import gc, hashlib, json, random, shutil, subprocess, sys, time, traceback
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
pd.options.display.float_format = "{:.3f}".format

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

REPO_URL = "https://github.com/TR-morph-retrieval/turkish-morph-retrieval.git"
ROOT = Path("/content/turkish-morph-retrieval")

def run_git(command):
    result = subprocess.run(command, text=True, capture_output=True)
    if result.returncode != 0:
        raise RuntimeError("Public GitHub deposuna erisilemedi. Runtime internet baglantisini ve REPO_URL degerini kontrol edin.\nGit: " + result.stderr[-800:])

if (ROOT / "test/evaluation.py").exists():
    run_git(["git", "-C", str(ROOT), "pull", "--ff-only"])
else:
    if ROOT.exists(): shutil.rmtree(ROOT)  # yalnız yarım kalmış Colab clone'u
    run_git(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)])
sys.path.insert(0, str(ROOT))

from sentence_transformers import SentenceTransformer
from test.evaluation import (
    EVALUATION_API_VERSION, FULL_CORPUS_RECALL_KS, artifact_baseline_summaries,
    bootstrap_ci, closed_qrels, evaluate_run, load_items, score_encoder,
)
assert EVALUATION_API_VERSION == "3.2", "Repo eski; yeni notebooku açıp runtime'ı yeniden başlatın."
GIT_COMMIT = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
print("repo:", ROOT); print("git commit:", GIT_COMMIT)
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
DATA_FILE = ROOT / "test/data/pilot_verisi_5.json"
ITEMS = load_items(DATA_FILE)
assert len(ITEMS) == 5, f"5 family bekleniyordu, {len(ITEMS)} bulundu"
assert all(len(item["candidates"]) == 11 for item in ITEMS)
assert all(Counter(c["role"] for c in item["candidates"]) == {"positive": 1, "hard_negative": 8, "easy_negative": 2} for item in ITEMS)
family_ids = [item["family_id"] for item in ITEMS]
corpus_ids = [c["id"] for item in ITEMS for c in item["candidates"]]
assert len(family_ids) == len(set(family_ids))
assert len(corpus_ids) == len(set(corpus_ids)) == 55

DATA_SHA256 = hashlib.sha256(DATA_FILE.read_bytes()).hexdigest()
FULL_CORPUS_DEPTH = len(corpus_ids)
OUTPUT_DIR = Path("/content/morph_eval_pilot5") if Path("/content").exists() else ROOT / "test/results/morph_eval_pilot5"
CACHE_DIR = OUTPUT_DIR / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
BATCH_SIZE = 16
N_BOOT = 2_000
print(f"family={len(ITEMS)} | candidate={FULL_CORPUS_DEPTH} | sha256={DATA_SHA256}")
print("source:", DATA_FILE.relative_to(ROOT))

## Ucuz baseline'lar

Bunlar veri artefaktını görmek içindir; 5 örnekte kesin sonuç çıkarılmaz.

In [ ]:
QRELS = closed_qrels(ITEMS)
artifact_summaries = artifact_baseline_summaries(ITEMS)
ARTIFACT_DF = pd.DataFrame(artifact_summaries).T.sort_values("recall@1", ascending=False)
display(ARTIFACT_DF[["recall@1", "recall@3", "mrr@10", "ndcg@10", "mean_rank"]].round(3))
print("closed-family chance R@1:", round(1/11, 4))

## Beş encoder

A100 için tamamı açıktır. Her model sırayla yüklenir ve sonra GPU belleği temizlenir.

In [ ]:
MODEL_SPECS = [
    {"name": "e5-large", "repo": "intfloat/multilingual-e5-large", "query_prefix": "query: ", "document_prefix": "passage: ", "enabled": True},
    {"name": "bge-m3", "repo": "BAAI/bge-m3", "query_prefix": "", "document_prefix": "", "enabled": True},
    {"name": "modernbert-tr", "repo": "ytu-ce-cosmos/modernbert-tr-embed", "query_prefix": "", "document_prefix": "", "enabled": True},
    {"name": "trmteb-ft-110m", "repo": "trmteb/turkish-embedding-model-fine-tuned", "query_prefix": "", "document_prefix": "", "enabled": True},
    {"name": "qwen3-8b", "repo": "Qwen/Qwen3-Embedding-8B", "query_prefix": "", "document_prefix": "", "enabled": True, "dtype": "float16"},
]
display(pd.DataFrame(MODEL_SPECS)[["name", "repo", "enabled"]])

In [ ]:
EVAL_SHA256 = hashlib.sha256((ROOT / "test/evaluation.py").read_bytes()).hexdigest()

def cache_path(spec):
    payload = json.dumps({"spec": spec, "data": DATA_SHA256, "eval": EVAL_SHA256}, sort_keys=True)
    return CACHE_DIR / f"{spec['name']}-{hashlib.sha256(payload.encode()).hexdigest()[:12]}.json"

def load_model(spec):
    kwargs = {"trust_remote_code": True}
    if spec.get("dtype") and torch.cuda.is_available():
        kwargs["model_kwargs"] = {"torch_dtype": getattr(torch, spec["dtype"])}
    model = SentenceTransformer(spec["repo"], **kwargs)
    try: model.default_prompt_name = None
    except Exception: pass
    return model

def score_with_backoff(model, spec):
    batch = BATCH_SIZE
    while True:
        try:
            return score_encoder(
                model, ITEMS, spec.get("query_prefix", ""), spec.get("document_prefix", ""),
                batch_size=batch, include_full_run=True, full_run_depth=FULL_CORPUS_DEPTH,
            ), batch
        except torch.cuda.OutOfMemoryError:
            if batch == 1: raise
            batch = max(1, batch // 2); gc.collect(); torch.cuda.empty_cache()
            print(f"CUDA OOM; batch_size={batch} ile yeniden deneniyor")

def run_model(spec):
    path = cache_path(spec)
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8")), {"model": spec["name"], "cached": True}
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter(); model = load_model(spec); load_seconds = time.perf_counter() - started
    dimension = model.get_sentence_embedding_dimension()
    started = time.perf_counter(); result, batch = score_with_backoff(model, spec); score_seconds = time.perf_counter() - started
    peak_gb = torch.cuda.max_memory_allocated()/1e9 if torch.cuda.is_available() else 0.0
    path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    del model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return result, {"model": spec["name"], "cached": False, "dimension": dimension, "batch_size": batch, "load_seconds": load_seconds, "score_seconds": score_seconds, "peak_gpu_gb": peak_gb}

In [ ]:
RESULTS, RUNTIME_ROWS, MODEL_ERRORS = {}, [], []
for spec in [spec for spec in MODEL_SPECS if spec["enabled"]]:
    print("\n===", spec["name"], "===")
    try:
        RESULTS[spec["name"]], runtime = run_model(spec); RUNTIME_ROWS.append(runtime)
    except Exception as exc:
        MODEL_ERRORS.append({"model": spec["name"], "error": repr(exc), "traceback": traceback.format_exc()})
        print("ATLANDI:", repr(exc)); gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
assert RESULTS, "Hiçbir model tamamlanmadı; MODEL_ERRORS çıktısına bakın."
display(pd.DataFrame(RUNTIME_ROWS))
if MODEL_ERRORS: display(pd.DataFrame(MODEL_ERRORS)[["model", "error"]])

## Ana metrikler

İlk tablo hızlı okumak içindir; ikinci tablo evaluation kodunun ürettiği diğer metrikleri de gösterir.

In [ ]:
SUMMARY_DF = pd.DataFrame({name: result["summary"] for name, result in RESULTS.items()}).T
focus = ["recall@1", "recall@3", "mean_rank", "pairwise_hard_accuracy", "contrast_consistency", "mrr@10", "ndcg@10"]
display(SUMMARY_DF[focus].sort_values("recall@1", ascending=False).round(3))
print("Tüm closed-family metrikleri:")
display(SUMMARY_DF.sort_values("recall@1", ascending=False).round(3))

## Gold sıraları ve bütün 11 adayın sırası

`gold_rank_11` kontrollü testtir. `gold_rank_full`, query'nin bütün 55-belge corpusundaki
sırasıdır. `RANKING_DF` her query için kendi 11 adayının tamamını listeler.

In [ ]:
item_by_id = {item["family_id"]: item for item in ITEMS}
gold_rank_rows, ranking_rows = [], []
for model_name, result in RESULTS.items():
    per_query = {row["query_id"]: row for row in result["per_query"]}
    for query_id, own_ranking in result["run"].items():
        item = item_by_id[query_id]; candidates = {c["id"]: c for c in item["candidates"]}
        full_ranking = result["full_run"][query_id]
        gold_rank_rows.append({
            "model": model_name, "family_id": query_id, "target_feature": item["target_feature"],
            "query": item["query"], "gold_rank_11": own_ranking.index(item["gold_id"])+1,
            "gold_rank_full": full_ranking.index(item["gold_id"])+1,
            "top_role": candidates[own_ranking[0]]["role"], "top_subtype": candidates[own_ranking[0]]["subtype"],
            "hard_rank": per_query[query_id]["hard_rank"],
        })
        for rank, candidate_id in enumerate(own_ranking, start=1):
            candidate = candidates[candidate_id]
            ranking_rows.append({
                "model": model_name, "family_id": query_id, "rank_11": rank,
                "is_gold": candidate_id == item["gold_id"], "role": candidate["role"],
                "subtype": candidate["subtype"], "score": result["scores"][query_id][candidate_id],
                "query": item["query"], "candidate_text": candidate["text"],
            })
GOLD_RANKS_DF = pd.DataFrame(gold_rank_rows).sort_values(["model", "gold_rank_11"], ascending=[True, False])
RANKING_DF = pd.DataFrame(ranking_rows).sort_values(["model", "family_id", "rank_11"])
display(GOLD_RANKS_DF)
display(RANKING_DF)

## 55-belge full-corpus retrieval ve bootstrap CI

In [ ]:
full_summaries, ci_rows = {}, []
for model_name, result in RESULTS.items():
    full_summaries[model_name] = evaluate_run(QRELS, result["full_run"], recall_ks=FULL_CORPUS_RECALL_KS)[0]
    for metric in ["recall@1", "pairwise_hard_accuracy", "mrr@10", "ndcg@10"]:
        values = [float(row[metric]) for row in result["per_query"]]
        low, high = bootstrap_ci(values, n_boot=N_BOOT, seed=SEED)
        ci_rows.append({"model": model_name, "metric": metric, "mean": np.mean(values), "ci_low": low, "ci_high": high, "n": len(values)})
FULL_CORPUS_DF = pd.DataFrame(full_summaries).T
CI_DF = pd.DataFrame(ci_rows)
display(FULL_CORPUS_DF[["recall@1", "recall@3", "recall@10", "recall@50", "mrr@10", "ndcg@10"]].round(3))
print(f"Full corpus: {len(ITEMS)} query, {FULL_CORPUS_DEPTH} ortak belge, query başına tek gold.")
display(CI_DF.round(3))

## Sonuçları indir

In [ ]:
SUMMARY_DF.to_csv(OUTPUT_DIR / "encoder_summary.csv", index_label="model")
GOLD_RANKS_DF.to_csv(OUTPUT_DIR / "gold_ranks_11_and_full.csv", index=False)
RANKING_DF.to_csv(OUTPUT_DIR / "all_candidate_rankings_11.csv", index=False)
FULL_CORPUS_DF.to_csv(OUTPUT_DIR / "full_corpus_retrieval.csv", index_label="model")
ARTIFACT_DF.to_csv(OUTPUT_DIR / "artifact_baselines.csv", index_label="baseline")
CI_DF.to_csv(OUTPUT_DIR / "bootstrap_ci_pilot.csv", index=False)
pd.DataFrame(RUNTIME_ROWS).to_csv(OUTPUT_DIR / "runtime.csv", index=False)
(OUTPUT_DIR / "model_errors.json").write_text(json.dumps(MODEL_ERRORS, ensure_ascii=False, indent=2), encoding="utf-8")
metadata = {"dataset_mode": "pilot5", "dataset_file": str(DATA_FILE.relative_to(ROOT)), "git_commit": GIT_COMMIT, "dataset_sha256": DATA_SHA256, "model_specs": MODEL_SPECS, "warning": "pilot_not_paper_result"}
(OUTPUT_DIR / "run_metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
archive = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR)
print("zip:", archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass